# IDM — Weekly Summary Generation (Ollama / Qwen3.5-4B, non-thinking)

Run this once a week to (re)generate the dated archive in `data/summaries/` (and `data/summary_latest.txt`), which the website's Summary page
reads on load. It calls your **LAN Ollama** server using the `/api/generate` route in
**non-thinking** mode, and feeds the model a compact, purely numeric data context so it can't
invent figures.

This mirrors the logic in the website's `assets/ai/idm-ai.js` (same context, same prompt), so the
cached file and the in-browser “Regenerate” button produce consistent summaries.


## 1. Configuration


In [ ]:
import os, json, re
iso = ts['date'].iloc[-1].strftime('%Y-%m-%d')
header = f"# India Drought Monitor \u2014 National Summary for week ending {iso}\n"

# 1) write the dated archive file the website reads from
arch_dir = f'{REPO_DIR}/data/summaries'
os.makedirs(arch_dir, exist_ok=True)
arch_path = f'{arch_dir}/summary_{iso}.txt'
with open(arch_path, 'w') as f: f.write(header + '\n' + summary + '\n')

# 2) also refresh summary_latest.txt (handy for any latest-only consumers)
with open(f'{REPO_DIR}/data/summary_latest.txt', 'w') as f: f.write(header + '\n' + summary + '\n')

# 3) rebuild the archive manifest (newest first) so the date picker shows the new week
months = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
items = []
for fn in os.listdir(arch_dir):
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})', fn)
    if fn.endswith('.txt') and m:
        y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        items.append({'file': fn, 'date': f'{y:04d}-{mo:02d}-{d:02d}', 'label': f'{months[mo]} {d}, {y}'})
items.sort(key=lambda x: x['date'], reverse=True)
with open(f'{arch_dir}/index.json', 'w') as f: json.dump({'summaries': items}, f, indent=2)

print('wrote', arch_path)
print('archive now has', len(items), 'summaries')
print(open(arch_path).read())

## 2. Load the data (national time series + per-state latest week)

Same files and the same per-state computation the website uses.


In [ ]:
import os, io, json, requests, pandas as pd, numpy as np
from collections import defaultdict

def read_text(rel):
    if USE_GITHUB_RAW:
        r = requests.get(f'{RAW_BASE}/{rel}', timeout=60); r.raise_for_status(); return r.text
    with open(os.path.join(REPO_DIR, rel)) as f: return f.read()

# national time series: year month day %Normal D0+ D1+ D2+ D3+ D4 (D0..D4 CUMULATIVE)
ts = pd.read_csv(io.StringIO(read_text('data/India_Drought_Area_Timeseries.txt')), sep=r'\s+', header=None,
                 names=['year','month','day','normal','d0','d1','d2','d3','d4'])
ts['date'] = pd.to_datetime(ts[['year','month','day']]); ts = ts.sort_values('date').reset_index(drop=True)
latest = ts['date'].iloc[-1]; latest_compact = latest.strftime('%Y%m%d')
print('weeks:', len(ts), '| latest:', latest.date())

In [ ]:
# per-state breakdown for the latest week (same method as data-tables / idm-ai.js)
state_grid = pd.read_csv(io.StringIO(read_text('states_with_boundaries.csv')))
state_grid = state_grid[state_grid['value'].astype(int) >= 2].copy()
ID2NAME = {int(p['state_id']): p['name'] for p in json.loads(read_text('state_vector_boundaries.json'))}

g = pd.read_csv(io.StringIO(read_text(f'data/Drough_TS/CDI_{latest_compact}.txt')), sep=r'\s+', header=None, names=['lat','lng','val'])
g = g[pd.to_numeric(g['val'], errors='coerce').notna()].copy(); g['val'] = g['val'].astype(float)
mn, mx = g['val'].min(), g['val'].max()
idx = {(round(r.lat,3), round(r.lng,3)): r.val for r in g.itertuples()}
def nearest(lat,lng):
    for dla in (0,0.0625,-0.0625,0.125,-0.125):
        for dlo in (0,0.0625,-0.0625,0.125,-0.125):
            k=(round(lat+dla,3), round(lng+dlo,3))
            if k in idx: return idx[k]
    return None
def classify(v):
    n=(v-mn)/((mx-mn) or 1)
    return 'D4' if n<0.12 else 'D3' if n<0.25 else 'D2' if n<0.40 else 'D1' if n<0.55 else 'D0' if n<0.75 else 'None'
tally=defaultdict(lambda: defaultdict(int))
for r in state_grid.itertuples():
    v=nearest(float(r.lat),float(r.lng))
    if v is None: continue
    tally[int(r.value)][classify(v)] += 1; tally[int(r.value)]['tot'] += 1
states=[]
for sid,c in tally.items():
    if c['tot']<8: continue
    pc=lambda k: round(100*c.get(k,0)/c['tot'],1)
    states.append({'state':ID2NAME.get(sid,f'State {sid}'),'drought':round(100-pc('None'),1),
                   'd2':pc('D2'),'d3':pc('D3'),'d4':pc('D4')})
states.sort(key=lambda x:x['drought'], reverse=True)
print('worst:', [s['state'] for s in states[:5]])

## 3. Build the numeric context and call the model (non-thinking)


In [ ]:
def pct(x): return 'n/a' if x is None else f'{x:.1f}%'
cur, prev, month_ago = ts.iloc[-1], ts.iloc[-2], ts.iloc[-5]
delta = cur['d0'] - prev['d0']
trend = 'expanded' if delta>0.3 else 'contracted' if delta<-0.3 else 'held roughly steady'
worst = states[:6]; best = states[-4:][::-1]
ctx = []
ctx.append(f"Week ending: {cur['date'].date()}")
ctx.append('National area by drought class (cumulative, % of India):')
ctx.append(f"  Normal: {pct(cur['normal'])}; D0+: {pct(cur['d0'])}; D1+: {pct(cur['d1'])}; D2+: {pct(cur['d2'])}; D3+: {pct(cur['d3'])}; D4: {pct(cur['d4'])}")
ctx.append(f"Total drought area (D0+): this week {pct(cur['d0'])}, last week {pct(prev['d0'])}, ~1 month ago {pct(month_ago['d0'])} => drought {trend} ({delta:+.1f} pts week-on-week).")
ctx.append('Most-affected states (by % area in any drought):')
for s in worst: ctx.append(f"  {s['state']}: {pct(s['drought'])} in drought (D2+ {s['d2']+s['d3']+s['d4']:.1f}%, D3+ {s['d3']+s['d4']:.1f}%)")
ctx.append('Least-affected states: ' + ', '.join(f"{s['state']} ({pct(s['drought'])})" for s in best))
DATA_CONTEXT='\n'.join(ctx)
print(DATA_CONTEXT)

In [ ]:
SYSTEM = ('You are a hydroclimatology analyst writing the weekly national summary for the India '
  'Drought Monitor (IDM), produced by the Water and Climate Lab, IIT Gandhinagar. The IDM uses a '
  'Combined Drought Index (CDI) with six classes: Normal, D0 (Abnormally Dry), D1 (Moderate), '
  'D2 (Severe), D3 (Extreme), D4 (Exceptional). Write clear, factual, neutral prose for a general '
  'audience. CRITICAL: use ONLY the numbers provided. Do NOT invent figures, place names, dates, '
  'or trends not supported by the data.')

PROMPT = ("Here is this week's India Drought Monitor data:\n\n" + DATA_CONTEXT + "\n\n"
  'Note: the national class areas are CUMULATIVE (e.g. \'D2 or worse\' already includes D3 and D4).\n\n'
  'Write a concise national drought summary of about 150-180 words in plain paragraphs (no headings, '
  'no bullet points, no markdown), covering: (1) a one-sentence overview of national conditions this '
  'week, (2) the week-on-week trend, and (3) which regions/states are most affected and any notably better.')

def generate(prompt, system):
    body = {'model': MODEL, 'prompt': prompt, 'system': system, 'stream': False, 'think': False,
            'options': {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20, 'num_predict': 600}}
    r = requests.post(API_URL, json=body, timeout=300); r.raise_for_status()
    txt = r.json().get('response','').strip()
    if '</think>' in txt: txt = txt.split('</think>')[-1].strip()
    return txt

summary = generate(PROMPT, SYSTEM)
print(summary)

## 4. Save to `data/summary_latest.txt`

Commit this file to the repo (or your weekly job can `git commit && push`). The website reads it on
the Summary page.


In [ ]:
header = f"# India Drought Monitor — National Summary for week ending {ts['date'].iloc[-1].date()}\n"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f: f.write(header + '\n' + summary + '\n')
print('wrote', OUTPUT_PATH)
print(open(OUTPUT_PATH).read())

---
**Automation:** schedule this notebook (or `jupyter nbconvert --to script` it) to run weekly after
the new CDI grid lands, then commit `data/summary_latest.txt`. The website always shows the cached
file instantly and only calls the model live if a user clicks “Regenerate with AI”.
